<a href="https://colab.research.google.com/github/learning710/11239A090_Compiler_Design_Lab/blob/main/cd_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def first_follow(grammar, start_symbol):
    non_terminals = set(grammar.keys())
    terminals = set()
    for prods in grammar.values():
        for prod in prods:
            for symbol in prod:
                if symbol not in non_terminals and symbol != 'ε':
                    terminals.add(symbol)

    first = {nt: set() for nt in non_terminals}
    first.update({t: {t} for t in terminals})
    first['ε'] = {'ε'}

    # Compute FIRST sets
    while True:
        updated = False
        for nt, productions in grammar.items():
            for prod in productions:
                # Handle epsilon production for the non-terminal itself
                if prod == ('ε',):
                    if 'ε' not in first[nt]:
                        first[nt].add('ε')
                        updated = True
                    continue

                for symbol in prod:
                    # Add FIRST(symbol) to FIRST(nt)
                    if symbol in non_terminals:
                        for f in first[symbol]:
                            if f != 'ε' and f not in first[nt]:
                                first[nt].add(f)
                                updated = True
                    else: # symbol is a terminal
                        if symbol not in first[nt]:
                            first[nt].add(symbol)
                            updated = True

                    # Check if symbol is nullable
                    if symbol in non_terminals and 'ε' in first[symbol]:
                        continue # Continue to next symbol in production
                    else:
                        break # Symbol is not nullable, stop adding FIRST from this production
                else: # All symbols in prod are nullable, so add epsilon to FIRST(nt)
                    if 'ε' not in first[nt]:
                        first[nt].add('ε')
                        updated = True
        if not updated:
            break

    follow = {nt: set() for nt in non_terminals}
    follow[start_symbol].add('$') # Add end-of-input marker to start symbol's FOLLOW

    # Compute FOLLOW sets
    while True:
        updated = False
        for nt, productions in grammar.items():
            for prod in productions:
                for i, symbol in enumerate(prod):
                    if symbol in non_terminals:
                        # A -> alpha B beta
                        # FOLLOW(B) includes FIRST(beta) excluding epsilon
                        j = i + 1
                        while j < len(prod):
                            next_symbol = prod[j]
                            for f in first[next_symbol]:
                                if f != 'ε' and f not in follow[symbol]:
                                    follow[symbol].add(f)
                                    updated = True

                            if next_symbol in non_terminals and 'ε' in first[next_symbol]:
                                j += 1
                            else:
                                break
                        else: # All subsequent symbols are nullable, or no subsequent symbols
                            # FOLLOW(B) includes FOLLOW(A)
                            for f in follow[nt]:
                                if f not in follow[symbol]:
                                    follow[symbol].add(f)
                                    updated = True
        if not updated:
            break

    return first, follow


In [5]:
# Example Grammar
grammar = {
    'E': [('T', 'Ep')],
    'Ep': [('+', 'T', 'Ep'), ('ε',)],
    'T': [('F', 'Tp')],
    'Tp': [('*', 'F', 'Tp'), ('ε',)],
    'F': [('(', 'E', ')'), ('id',)]
}

start_symbol = 'E'

first, follow = first_follow(grammar, start_symbol)

print("FIRST Sets:")
for nt, s in first.items():
    print(f"FIRST({nt}) = {{ {', '.join(sorted(list(s)))} }}")

print("\nFOLLOW Sets:")
for nt, s in follow.items():
    print(f"FOLLOW({nt}) = {{ {', '.join(sorted(list(s)))} }}")

FIRST Sets:
FIRST(F) = { (, id }
FIRST(Tp) = { *, ε }
FIRST(T) = { (, id }
FIRST(E) = { (, id }
FIRST(Ep) = { +, ε }
FIRST()) = { ) }
FIRST(+) = { + }
FIRST(() = { ( }
FIRST(*) = { * }
FIRST(id) = { id }
FIRST(ε) = { ε }

FOLLOW Sets:
FOLLOW(F) = { $, ), *, + }
FOLLOW(Tp) = { $, ), + }
FOLLOW(T) = { $, ), + }
FOLLOW(E) = { $, ) }
FOLLOW(Ep) = { $, ) }
